# Modeling

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, pickle

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
os.makedirs('../outputs/figures', exist_ok=True)
os.makedirs('../outputs/models', exist_ok=True)

train_df = pd.read_csv('../data/processed/train.csv')
test_df  = pd.read_csv('../data/processed/test.csv')
with open('../data/processed/scaler_y.pkl','rb') as f: scaler_y = pickle.load(f)

feature_cols = [c for c in train_df.columns if c not in ['loan_amnt_scaled','loan_amnt_raw']]
X_train    = train_df[feature_cols].values
y_train_sc = train_df['loan_amnt_scaled'].values
X_test     = test_df[feature_cols].values
y_test_raw = test_df['loan_amnt_raw'].values
n_features = X_train.shape[1]

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}  |  TF: {tf.__version__}')

## Dummy Baseline

Predicts the training mean every time — the floor everything else needs to beat.

In [ ]:
dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train, y_train_sc)
y_pred_dm = scaler_y.inverse_transform(dummy.predict(X_test).reshape(-1,1)).flatten()

rmse = np.sqrt(mean_squared_error(y_test_raw, y_pred_dm))
print(f'Dummy  RMSE: ${rmse:,.0f}  R²: {r2_score(y_test_raw, y_pred_dm):.4f}')
with open('../outputs/models/dummy.pkl','wb') as f: pickle.dump(dummy, f)

## Linear Regression

Hard benchmark on tabular data. Coefficients also show which features matter.

In [ ]:
from sklearn.model_selection import train_test_split
import json

with open('../data/processed/feature_names.pkl','rb') as f: feature_names = pickle.load(f)
with open('../data/processed/scaler_X.pkl','rb') as f: scaler_X = pickle.load(f)

df_raw = pd.read_csv('../data/raw/credit_risk.csv')
df_raw['person_emp_length'] = df_raw['person_emp_length'].fillna(df_raw['person_emp_length'].median())
df_raw['loan_int_rate']     = df_raw['loan_int_rate'].fillna(df_raw.groupby('loan_grade')['loan_int_rate'].transform('median'))
cat_cols = ['person_home_ownership','loan_intent','loan_grade','cb_person_default_on_file']
df_enc = pd.get_dummies(df_raw, columns=cat_cols, drop_first=True)
y_all  = df_enc['loan_amnt']
X_all  = df_enc.drop(columns=['loan_amnt','loan_percent_income','loan_status'])
_, _, y_train_raw, y_test_raw2 = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

lr = LinearRegression()
lr.fit(X_train, y_train_raw)
y_pred_lr = lr.predict(X_test)

print(f'LR  RMSE: ${np.sqrt(mean_squared_error(y_test_raw2, y_pred_lr)):,.0f}  R²: {r2_score(y_test_raw2, y_pred_lr):.4f}')
with open('../outputs/models/linear_regression.pkl','wb') as f: pickle.dump(lr, f)

In [ ]:
coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': lr.coef_})
coef_df['abs_coef'] = np.abs(coef_df['Coefficient'])
coef_df = coef_df.sort_values('abs_coef', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#C62828' if c > 0 else '#1565C0' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Coefficient ($ per 1-std-dev)')
ax.set_title('Top 15 features — Linear Regression', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../outputs/figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Neural Network

Linear output (`Dense(1, activation='linear')`), MSE loss, StandardScaler on y.

In [ ]:
def build_model(n_features):
    model = Sequential([
        Dense(128, input_shape=(n_features,), activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(16, activation='relu'),
        Dense(1, activation='linear'),
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

nn = build_model(n_features)
nn.summary()

In [ ]:
callbacks = [
    EarlyStopping(patience=6, monitor='val_loss', restore_best_weights=True),
    ModelCheckpoint('../outputs/models/best_nn.keras', monitor='val_loss', save_best_only=True)
]

history = nn.fit(
    X_train, y_train_sc,
    epochs=60, batch_size=64, validation_split=0.15,
    callbacks=callbacks, verbose=1
)
print(f'Stopped at epoch {len(history.history["loss"])}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ep = range(1, len(history.history['loss'])+1)

axes[0].plot(ep, history.history['loss'],     label='Train', lw=2)
axes[0].plot(ep, history.history['val_loss'], label='Val',   lw=2, linestyle='--')
axes[0].set_title('MSE loss (scaled y)', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE'); axes[0].legend()

axes[1].plot(ep, history.history['mae'],     label='Train MAE', lw=2)
axes[1].plot(ep, history.history['val_mae'], label='Val MAE',   lw=2, linestyle='--')
axes[1].set_title('MAE (scaled)', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MAE'); axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## TFLite Export

BatchNorm in TF 2.16 needs the old converter path — export to SavedModel first.

In [ ]:
best_nn = tf.keras.models.load_model('../outputs/models/best_nn.keras')

# Verify output layer is linear (regression)
last = best_nn.layers[-1]
print(f'Output: {last.name}  activation: {last.activation.__name__}  units: {last.units}')

# Sample prediction
pred_sc = best_nn.predict(X_test[:1], verbose=0).flatten()[0]
pred_usd = scaler_y.inverse_transform([[pred_sc]])[0][0]
print(f'Sample prediction: ${pred_usd:,.0f}  (actual: ${y_test_raw[0]:,.0f})')

print(f'scaler_y mean={scaler_y.mean_[0]:,.2f}  scale={scaler_y.scale_[0]:,.2f}')

In [ ]:
saved_path = '/tmp/loan_saved_model'
best_nn.export(saved_path)

converter = tf.lite.TFLiteConverter.from_saved_model(saved_path)
converter.experimental_new_converter = False
tflite_model = converter.convert()

tflite_path = '../outputs/models/model.tflite'
with open(tflite_path, 'wb') as f: f.write(tflite_model)
print(f'Saved: {tflite_path}  ({len(tflite_model)/1024:.1f} KB)')